# Entrenar Stable Diffusion con LoRA
## Usando las ilustraciones recolectadas por el equipo

Este notebook procesa todas las imágenes del equipo y afina Stable Diffusion con LoRA para que aprenda nuestro estilo visual. Solo necesitas cambiar `CARPETA_RAIZ` al inicio.

**Requisitos:** Colab con GPU (T4 o superior).

In [ ]:
# Paso 1 — Instalación (solo la primera vez)
!pip install diffusers peft torchvision transformers accelerate pillow -q

# Paso 2 — Configuración
Cambia `CARPETA_RAIZ` a donde tengas la carpeta `IMAGENES_CUENTOS`.
Si está en Drive, primero monta Drive con la celda de abajo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import csv
import hashlib
from pathlib import Path
from PIL import Image

CARPETA_RAIZ = "/content/drive/MyDrive/IMAGENES_CUENTOS"  # <-- CAMBIA ESTO

# Mapeo de carpetas a temáticas (hardcodeado del Drive del equipo)
MAPEO_CARPETAS = {
    "animales_yaretzi": "animales",
    "animales_yaretzi_mexica": "animales",
    "espacio_alejandro_rodea": "espacio",
    "espacio_cristobal_sanchez": "espacio",
    "dinosaurios_y_prehistoria_munguia_cesar": "dinosaurios y prehistoria",
    "prehistoria_dinosarurios_marco-nieves": "dinosaurios y prehistoria",
    "fantasmas_y_misterio_evelyne_rojas": "fantasmas y misterio",
    "fantasmas_y_misterio_Ingrid_Salceda": "fantasmas y misterio",
    "fantasmas y misterio_ Ingrid Salceda": "fantasmas y misterio",
    "heroes_y_aventuras_reynoso_valeria": "heroes y aventuras",
    "magia_y_brujas_karen_flores": "magia y brujas",
    "mis_ilustraciones_moises_cordero": "magia y brujas",
    "mar_y_oceano_Dario_Fuentes": "mar y oceano",
    "mar_y_oceano_fernanda_hernandez": "mar y oceano",
    "monstruos_y_criaturas_fernanda_garcia": "monstruos y criaturas",
    "naturaleza_y_bosques_ernesto_guevara": "naturaleza y bosques",
    "naturaleza_y_bosques_Raul_Hernandez": "naturaleza y bosques",
    "piratas_gabriela_cervantes": "piratas",
    "piratas_Reyna_Alvarez": "piratas",
    "princesas_y_castillos_dayan_garcia": "princesas y castillos",
    "princesas_y_castillos_grisel": "princesas y castillos",
    "mis_ilustraciones": "princesas y castillos",
    "mis_ilustraciones_grisel": "princesas y castillos",
    "robots_y_tecnologia_axel_mendoza": "robots y tecnologia",
    "robots_y_tecnologia_karla_melgarejo": "robots y tecnologia",
    "mis_ilustraciones_antonio": "animales",
    "mis_ilustraciones_axel_mendoza": "robots y tecnologia",
    "mis_ilustraciones_cristobal_sanc": "espacio",
    "mis_ilustraciones_ernesto_guevara": "naturaleza y bosques",
    "mis_ilustraciones_fernanda_garcia": "monstruos y criaturas",
    "mis_ilustraciones_fernanda_hernandez": "mar y oceano",
    "mis_ilustraciones_fuentes_gonzalez": "mar y oceano",
    "mis_ilustraciones_gabriela_cervantes": "piratas",
    "mis_ilustraciones_Hernandez_Raul": "naturaleza y bosques",
    "mis_ilustraciones_karen_flores": "magia y brujas",
    "mis_ilustraciones_karla_melgarejo": "robots y tecnologia",
    "mis_ilustraciones_munguia_cesar": "dinosaurios y prehistoria",
    "mis_ilustraciones_nieves_bartolo": "dinosaurios y prehistoria",
    "mis_ilustraciones_Reyna_Alvarez": "piratas",
    "mis_ilustraciones_reynoso_valeria": "heroes y aventuras",
    "mis_ilustraciones_rojas_evelyne": "fantasmas y misterio",
    "mis_ilustraciones_yaretzi_mexica": "animales",
    "alejandro_rodea": "espacio",
}

EXTENSIONES_VALIDAS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
LADO_MINIMO = 64

# Paso 3 — Escanear carpetas
Recorre todas las subcarpetas, identifica la temática de cada una y lista las imágenes.

In [ ]:
def escanear_imagenes(carpeta_raiz):
    resultados = []
    carpetas_no_mapeadas = []

    for entrada in sorted(os.listdir(carpeta_raiz)):
        ruta_sub = os.path.join(carpeta_raiz, entrada)
        if not os.path.isdir(ruta_sub):
            continue

        tematica = None
        for clave, tem in MAPEO_CARPETAS.items():
            if entrada == clave or entrada.startswith(clave):
                tematica = tem
                break

        if tematica is None:
            carpetas_no_mapeadas.append(entrada)
            continue

        conteo = 0
        for archivo in os.listdir(ruta_sub):
            ext = os.path.splitext(archivo)[1].lower()
            if ext not in EXTENSIONES_VALIDAS:
                continue
            ruta_img = os.path.join(ruta_sub, archivo)
            resultados.append({"ruta": ruta_img, "tematica": tematica, "carpeta": entrada})
            conteo += 1

        print(f"  {entrada}: {conteo} imágenes → {tematica}")

    if carpetas_no_mapeadas:
        print(f"\nCarpetas NO mapeadas (ignoradas): {carpetas_no_mapeadas}")

    return resultados

print("Escaneando carpetas...\n")
imagenes = escanear_imagenes(CARPETA_RAIZ)
print(f"\nTotal: {len(imagenes)} imágenes")

# Paso 4 — Validar y deduplicar
Descarta imágenes corruptas o muy chicas y elimina duplicadas por hash.

In [ ]:
def validar_y_deduplicar(imagenes):
    validas = []
    hashes_vistos = set()
    descartadas = {"corrupta": 0, "muy_chica": 0, "duplicada": 0}

    for i, img_info in enumerate(imagenes):
        if i % 2000 == 0:
            print(f"  Procesando {i}/{len(imagenes)}...")

        ruta = img_info["ruta"]
        try:
            with Image.open(ruta) as img:
                ancho, alto = img.size
                img.verify()
        except Exception:
            descartadas["corrupta"] += 1
            continue

        if ancho < LADO_MINIMO or alto < LADO_MINIMO:
            descartadas["muy_chica"] += 1
            continue

        h = hashlib.sha1(open(ruta, "rb").read()).hexdigest()
        if h in hashes_vistos:
            descartadas["duplicada"] += 1
            continue
        hashes_vistos.add(h)

        img_info["hash"] = h
        validas.append(img_info)

    print(f"\nResultado:")
    print(f"  Válidas y únicas: {len(validas)}")
    print(f"  Corruptas: {descartadas['corrupta']}")
    print(f"  Muy chicas: {descartadas['muy_chica']}")
    print(f"  Duplicadas: {descartadas['duplicada']}")
    return validas

print("Validando y deduplicando...\n")
imagenes_validas = validar_y_deduplicar(imagenes)

# Paso 5 — Resumen por temática

In [ ]:
from collections import Counter

conteo = Counter(img["tematica"] for img in imagenes_validas)
print("Imágenes por temática:\n")
for tem, n in sorted(conteo.items()):
    print(f"  {tem:<30} {n:>6}")
print(f"\n  {'TOTAL':<30} {sum(conteo.values()):>6}")

# Paso 6 — Preparar dataset
Redimensiona las imágenes a 512×512 y arma los prompts.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

LADO = 512

class ImagenesEstiloDataset(Dataset):
    def __init__(self, imagenes_info):
        self.imagenes = imagenes_info
        self.transform = transforms.Compose([
            transforms.Resize((LADO, LADO)),
            transforms.CenterCrop(LADO),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.imagenes)

    def __getitem__(self, i):
        info = self.imagenes[i]
        img = Image.open(info["ruta"]).convert("RGB")
        tensor = self.transform(img)
        prompt = f"ilustracion plana a color sobre {info['tematica']}"
        return {"pixel_values": tensor, "prompt": prompt}

dataset = ImagenesEstiloDataset(imagenes_validas)
print(f"Dataset listo: {len(dataset)} imágenes a {LADO}x{LADO}")
print(f"Prompt ejemplo: '{dataset[0]['prompt']}'")

# Paso 7 — Cargar Stable Diffusion y aplicar LoRA

In [ ]:
from diffusers import StableDiffusionPipeline, DDPMScheduler
from peft import LoraConfig, get_peft_model

MODELO_BASE = "stable-diffusion-v1-5/stable-diffusion-v1-5"
RANGO_LORA = 4
LR = 1e-4
PASOS = 1500  # ajusta segun cuanto tiempo tengas

print(f"Cargando {MODELO_BASE}...")
pipe = StableDiffusionPipeline.from_pretrained(MODELO_BASE, torch_dtype=torch.float32)
unet = pipe.unet.to("cuda")
vae = pipe.vae.to("cuda")
tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder.to("cuda")
noise_scheduler = DDPMScheduler.from_pretrained(MODELO_BASE, subfolder="scheduler")

lora_config = LoraConfig(
    r=RANGO_LORA,
    lora_alpha=RANGO_LORA,
    target_modules=["to_q", "to_v", "to_k", "to_out.0"],
    lora_dropout=0.05,
)
unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

vae.requires_grad_(False)
text_encoder.requires_grad_(False)
print("Modelo listo para entrenar.")

# Paso 8 — Entrenar

In [ ]:
cargador = DataLoader(dataset, batch_size=1, shuffle=True)
optimizador = torch.optim.AdamW(unet.parameters(), lr=LR)

unet.train()
paso = 0
perdidas = []

print(f"Entrenando {PASOS} pasos...\n")

while paso < PASOS:
    for lote in cargador:
        if paso >= PASOS:
            break

        with torch.no_grad():
            latentes = vae.encode(lote["pixel_values"].to("cuda")).latent_dist.sample() * 0.18215

        ruido = torch.randn_like(latentes)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                  (latentes.shape[0],), device="cuda").long()
        latentes_ruidosos = noise_scheduler.add_noise(latentes, ruido, timesteps)

        with torch.no_grad():
            tokens = tokenizer(lote["prompt"], padding="max_length",
                               max_length=tokenizer.model_max_length,
                               truncation=True, return_tensors="pt").input_ids.to("cuda")
            encoder_hidden = text_encoder(tokens)[0]

        prediccion = unet(latentes_ruidosos, timesteps, encoder_hidden).sample
        perdida = torch.nn.functional.mse_loss(prediccion, ruido)

        optimizador.zero_grad()
        perdida.backward()
        optimizador.step()

        paso += 1
        perdidas.append(perdida.item())

        if paso % 100 == 0 or paso == 1:
            print(f"  Paso {paso}/{PASOS}  pérdida={perdida.item():.4f}")

print("\nEntrenamiento terminado.")

# Paso 9 — Gráfica de pérdida

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(perdidas)
plt.title("Pérdida durante el entrenamiento LoRA")
plt.xlabel("Paso")
plt.ylabel("Pérdida (MSE)")
plt.tight_layout()
plt.show()

# Paso 10 — Guardar pesos LoRA

In [ ]:
CARPETA_SALIDA = "/content/drive/MyDrive/modelos/estilo_lora"  # <-- CAMBIA SI QUIERES
os.makedirs(CARPETA_SALIDA, exist_ok=True)
unet.save_pretrained(CARPETA_SALIDA)
with open(os.path.join(CARPETA_SALIDA, "config_base.txt"), "w") as f:
    f.write(MODELO_BASE)
print(f"Pesos LoRA guardados en {CARPETA_SALIDA}")

# Paso 11 — Probar generación
Genera imágenes de prueba para verificar que el estilo se aprendió.

In [ ]:
from diffusers import StableDiffusionPipeline
from peft import PeftModel

pipe_test = StableDiffusionPipeline.from_pretrained(
    MODELO_BASE, torch_dtype=torch.float16, safety_checker=None
)
pipe_test.unet = PeftModel.from_pretrained(pipe_test.unet, CARPETA_SALIDA)
pipe_test = pipe_test.to("cuda")

tematicas_prueba = [
    "ilustracion plana a color sobre espacio",
    "ilustracion plana a color sobre piratas",
    "ilustracion plana a color sobre animales",
    "ilustracion plana a color sobre fantasmas y misterio",
]

fig, axes = plt.subplots(1, len(tematicas_prueba), figsize=(20, 5))
for ax, prompt in zip(axes, tematicas_prueba):
    img = pipe_test(prompt=prompt,
                    negative_prompt="fotografia, realista, 3d, texto, marca de agua",
                    num_inference_steps=30, guidance_scale=7.5).images[0]
    ax.imshow(img)
    tema_corto = prompt.replace("ilustracion plana a color sobre ", "")
    ax.set_title(tema_corto)
    ax.axis("off")
plt.suptitle("Prueba de generación con LoRA entrenado")
plt.tight_layout()
plt.show()

print("Si las imágenes tienen el estilo plano/caricatura de nuestras ilustraciones, el LoRA funcionó.")